In [1]:
import os
os.chdir("../../..")

In [2]:
import torch
from torch.utils.data import Dataset
import pickle
import numpy as np
from tqdm import tqdm
from datasets.Bacteria_ID.config import ATCC_GROUPINGS,STRAINS,antibiotics

In [3]:
class Bacteria_Dataset(Dataset):
    def __init__(self,X_path,y_path,num_classes=30):
        """
        X_path is a string containing the path to the spectra
        y_path is a string containing the path to the labels for the spectra
        num_classes is an integer corresponding to an 8 class or 30 class problem 
        """
        super().__init__()
        self.X = np.load(X_path)
        self.y = np.load(y_path)
        self.num_classes = num_classes
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self,index):
        data = torch.Tensor(self.X[index]).unsqueeze(0) #of shape (1,1000)
        data = (data-data.min())/(data.max()-data.min())
        label_30 = self.y[index]
        label_8 = ATCC_GROUPINGS[label_30]
        return data,int(label_30),int(label_8)

In [4]:
ref_set = Bacteria_Dataset("datasets/Bacteria_ID/X_reference.npy","datasets/Bacteria_ID/y_reference.npy")

strain_count_ref = {i:0 for i in range(30)}
antibiotic_count_ref = {i:0 for i in range(8)}

loop = tqdm(ref_set)

for _,y30,y8 in loop:
    strain_count_ref[y30]+=1 
    antibiotic_count_ref[y8]+=1 

100%|██████████| 60000/60000 [00:01<00:00, 32620.82it/s]


In [5]:
fine_set = Bacteria_Dataset("datasets/Bacteria_ID/X_finetune.npy","datasets/Bacteria_ID/y_finetune.npy")

strain_count_fine = {i:0 for i in range(30)}
antibiotic_count_fine = {i:0 for i in range(8)}

loop = tqdm(fine_set)

for _,y30,y8 in loop:
    strain_count_fine[y30]+=1 
    antibiotic_count_fine[y8]+=1 

100%|██████████| 3000/3000 [00:00<00:00, 33023.07it/s]


In [6]:
test_set = Bacteria_Dataset("datasets/Bacteria_ID/X_test.npy","datasets/Bacteria_ID/y_test.npy")

strain_count_test = {i:0 for i in range(30)}
antibiotic_count_test = {i:0 for i in range(8)}

loop = tqdm(test_set)

for _,y30,y8 in loop:
    strain_count_test[y30]+=1 
    antibiotic_count_test[y8]+=1 


100%|██████████| 3000/3000 [00:00<00:00, 34098.28it/s]


In [7]:
for k,v in strain_count_test.items():
    print(f"{k} & {STRAINS[k]} & {antibiotics[ATCC_GROUPINGS[k]]} & {strain_count_ref[k]} & {strain_count_fine[k]} & {v} \\\\")

0 & C. albicans & Caspofungin & 2000 & 100 & 100 \\
1 & C. glabrata & Caspofungin & 2000 & 100 & 100 \\
2 & K. aerogenes & Meropenem & 2000 & 100 & 100 \\
3 & E. coli 1 & Meropenem & 2000 & 100 & 100 \\
4 & E. coli 2 & Meropenem & 2000 & 100 & 100 \\
5 & E. faecium & Daptomycin & 2000 & 100 & 100 \\
6 & E. faecalis 1 & Penicillin & 2000 & 100 & 100 \\
7 & E. faecalis 2 & Penicillin & 2000 & 100 & 100 \\
8 & E. cloacae & Meropenem & 2000 & 100 & 100 \\
9 & K. pneumoniae 1 & Meropenem & 2000 & 100 & 100 \\
10 & K. pneumoniae 2 & Meropenem & 2000 & 100 & 100 \\
11 & P. mirabilis & Meropenem & 2000 & 100 & 100 \\
12 & P. aeruginosa 1 & TZP & 2000 & 100 & 100 \\
13 & P. aeruginosa 2 & TZP & 2000 & 100 & 100 \\
14 & MSSA 1 & Vancomycin & 2000 & 100 & 100 \\
15 & MSSA 3 & Vancomycin & 2000 & 100 & 100 \\
16 & MRSA 1 (isogenic) & Vancomycin & 2000 & 100 & 100 \\
17 & MRSA 2 & Vancomycin & 2000 & 100 & 100 \\
18 & MSSA 2 & Vancomycin & 2000 & 100 & 100 \\
19 & S. enterica & Ciprofloxacin & 2000

In [8]:
for k,v in antibiotic_count_test.items():
    print(f"{k} & {antibiotics[k]} & {antibiotic_count_ref[k]} & {antibiotic_count_fine[k]} & {v} \\\\")

0 & Meropenem & 16000 & 800 & 800 \\
1 & Ciprofloxacin & 2000 & 100 & 100 \\
2 & TZP & 4000 & 200 & 200 \\
3 & Vancomycin & 14000 & 700 & 700 \\
4 & Ceftriaxone & 4000 & 200 & 200 \\
5 & Penicillin & 14000 & 700 & 700 \\
6 & Daptomycin & 2000 & 100 & 100 \\
7 & Caspofungin & 4000 & 200 & 200 \\
